# 🎙️ Oula's AI voice for Alsama Words · صوت علا

This makes every word in the app in **Oula's voice**, copied from her own recordings
(free tool: [Chatterbox](https://github.com/resemble-ai/chatterbox), MIT licence). Oula gave permission for this copy of her voice.

### What to do (no recording needed)
1. Top menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Top menu **Runtime → Run all**. If Google asks "Run anyway?", click **Run anyway**.
3. After a few minutes you can **listen to 5 test words** below (step 3).
4. Keep this tab open and **only one Colab tab** (close any other). All words take about **1 hour**.
   Your words are saved in **My Drive → oula-ai-voice** as they are made.
5. At the end, **oula-ai-voice.zip** downloads by itself. **Send that file to Claude.** Done!

If you see **"Runtime disconnected"**: click **Reconnect**, then **Runtime → Run all** again. It continues where it stopped.


## 1 · Check the free GPU

In [ ]:
!nvidia-smi -L || echo '⚠️ No GPU. Runtime → Change runtime type → T4 GPU, then Runtime → Run all.'

## 1b · Keep the progress in your Google Drive
A window asks to connect Google Drive: choose your account and click **Allow / Continue**.
Every finished word is saved in **My Drive → oula-ai-voice**, so if Google disconnects, **Run all** again continues where it stopped.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['OUT_DIR'] = '/content/drive/MyDrive/oula-ai-voice'
os.makedirs(os.environ['OUT_DIR'] + '/clone', exist_ok=True)
print('Saving to My Drive → oula-ai-voice ✅', len(os.listdir(os.environ['OUT_DIR'] + '/clone')), 'words already done')


## 2 · Install the free voice tool (about 3 minutes)

In [ ]:
!pip -q install chatterbox-tts==0.1.7
print('Installed ✅')

In [ ]:
%%writefile /content/generate_voice.py
# Makes every Alsama Words word in Oula's AI voice (Chatterbox, free, MIT licence).
# Runs on Google Colab. Words that already have Oula's real recording are skipped.
import json, os, subprocess, sys, urllib.request, zipfile

import torch
import torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

RAW = "https://raw.githubusercontent.com/Oula-ALS/ALS/main/vocab"
BASE = os.environ.get("OUT_DIR", "/content/out")  # Google Drive folder, so progress survives a disconnect
OUT = f"{BASE}/clone"
ZIP = f"{BASE}/oula-ai-voice.zip"
os.makedirs(OUT, exist_ok=True)


def get(path):
    with urllib.request.urlopen(f"{RAW}/{path}") as r:
        return r.read()


with open("/content/oula-voice-prompt.wav", "wb") as f:
    f.write(get("tools/oula-voice-prompt.wav"))
recorded = set(json.loads(get("src/data/audio.json")).get("recorded", []))
words = []
for level in ["beginner", "intermediate", "advanced"]:
    words += json.loads(get(f"src/data/{level}.json"))["words"]
todo = [w for w in words if w["id"] not in recorded]
limit = int(os.environ.get("LIMIT", "0")) or len(todo)
todo = todo[:limit]
print(f"{len(todo)} words to make in Oula's AI voice ({len(recorded)} already have her real recording).")

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("⚠️  No GPU: this will be very slow. Runtime → Change runtime type → T4 GPU.")
model = ChatterboxMultilingualTTS.from_pretrained(device=device)
model.prepare_conditionals("/content/oula-voice-prompt.wav", exaggeration=0.5)


def text_for(w):
    forms = [f.replace("…", "").replace("ـ", "").strip() for f in w["alternatives"]]
    return "، ".join(f for f in forms if f)


def length_ok(seconds, text):
    # A wrong take is usually far too long (the model kept talking) or empty.
    letters = len(text.replace(" ", ""))
    return 0.25 <= seconds <= 1.2 + 0.3 * letters


problems = []
for i, w in enumerate(todo, 1):
    mp3 = f"{OUT}/{w['id']}.mp3"
    if os.path.exists(mp3):
        continue
    text = text_for(w)
    take = None
    for attempt in range(3):
        torch.manual_seed(1000 + attempt)
        wav = model.generate(text, language_id="ar", exaggeration=0.5, cfg_weight=0.5, temperature=0.7)
        seconds = wav.shape[-1] / model.sr
        if take is None or length_ok(seconds, text):
            take = wav
        if length_ok(seconds, text):
            break
    else:
        problems.append(f"{w['id']}\t{w['display']}\t{w['english']}")
    ta.save("/content/take.wav", take, model.sr)
    # Trim silence at both ends, even out loudness, small MP3.
    subprocess.run(
        ["ffmpeg", "-v", "error", "-y", "-i", "/content/take.wav", "-af",
         "silenceremove=start_periods=1:start_threshold=-45dB,areverse,"
         "silenceremove=start_periods=1:start_threshold=-45dB,areverse,loudnorm=I=-18:TP=-2",
         "-ar", "24000", "-ac", "1", "-b:a", "64k", mp3],
        check=True,
    )
    if i % 25 == 0 or i == len(todo):
        print(f"{i}/{len(todo)} done", flush=True)

with open(f"{BASE}/check-these.txt", "w") as f:
    f.write("Words that may sound wrong (listen to these first):\n" + "\n".join(problems) + "\n")
with zipfile.ZipFile(ZIP, "w") as z:
    for name in sorted(os.listdir(OUT)):
        z.write(f"{OUT}/{name}", f"clone/{name}")
    z.write(f"{BASE}/check-these.txt", "check-these.txt")
print(f"\n✅ {len(os.listdir(OUT))} words in {ZIP}. {len(problems)} may need a listen (check-these.txt).")


## 3 · Five test words: listen! 🎧

In [ ]:
!LIMIT=5 python /content/generate_voice.py
import os
from IPython.display import Audio, display
for name in sorted(os.listdir(os.environ['OUT_DIR'] + '/clone'))[:5]:
    print(name)
    display(Audio(os.environ['OUT_DIR'] + '/clone/' + name))

## 4 · All the other words (about 1 hour, keep this tab open)

In [ ]:
!python /content/generate_voice.py

## 5 · Download the file and send it to Claude

In [ ]:
import os
from google.colab import files
files.download(os.environ['OUT_DIR'] + '/oula-ai-voice.zip')
print('If the download does not start: My Drive → oula-ai-voice → oula-ai-voice.zip')
